# IFC and Speckle exchange round trip (mock mode)

This notebook runs AEC Model Bridge in mock mode: no Revit, no Windows, no network. It walks an exchange round trip through the same `call_tool` handler an MCP client reaches:

1. `snapshot_take` and `snapshot_compute`: element counts and a few properties of the (mock) model.
2. `plan_actions` with `revit_export` (format `ifc`): how a real export starts. Mock mode cannot write an IFC, so the plan stays `pending` and the notebook uses the repository's test fixture `tests/data/ids/tiny.ifc` instead.
3. `ifc_get_metadata`, `ifc_validate` and `ids_validator_validate_ifc`: read and check the IFC.
4. Compare the IFC with the snapshot: counts per class and one property.
5. `speckle_auth_status`, then `plan_actions` with `speckle_send_object`: how a send would start. Receiving needs a Speckle server and is skipped.

The IFC steps are skipped when `ifcopenshell` is not installed; the IDS step is skipped without `ifctester` (the `ids` extra). Nothing is approved or executed. See `docs/ifc-speckle-roundtrip.md` for what is mock and what is real.

Run it from `packages/mcp-server-revit` after `uv sync --all-extras`, for example with `uv run --with jupyter jupyter lab ../../examples/ifc-speckle-roundtrip.ipynb`. Each run writes to a new temporary workspace folder.

In [ ]:
import importlib
import json
import os
import shutil
import tempfile
from pathlib import Path

# Settings are read when the package is imported, so set them first.
os.environ["MCP_REVIT_MODE"] = "mock"
WORKSPACE = Path(tempfile.mkdtemp(prefix="aec-roundtrip-"))
os.environ["MCP_REVIT_WORKSPACE_DIR"] = str(WORKSPACE)

from revit_mcp_server import mcp_server
from revit_mcp_server.registry_factory import build_registry
from revit_mcp_server.tool_profiles import load_tool_profile

# The same wiring the server does at start-up (mcp_server.main), without stdio.
registry, approval, jobs, _modules, workspace = build_registry()
registry.freeze_tool_profile(load_tool_profile())
mcp_server.registry = registry
mcp_server.approval_provider = approval
mcp_server.job_manager = jobs
mcp_server.workspace_monitor = workspace


async def call(name, arguments=None):
    """Call a tool the way an MCP client does and return its JSON result."""
    out = await mcp_server.call_tool(name, arguments or {})
    if getattr(out, "isError", False):
        return out.structuredContent  # {"ok": false, "error": {"kind": ..., "message": ...}}
    if getattr(out, "structuredContent", None) is not None:
        return out.structuredContent  # tools with an outputSchema return it directly
    text = getattr(out, "content", out)[0].text
    return json.loads(text.split("Result:\n", 1)[1])


def importable(module):
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False


HAVE_IFC = importable("ifcopenshell")
HAVE_IDS = HAVE_IFC and importable("ifctester")
TOOLS = {tool.name for tool in await mcp_server.list_tools()}
report = {"ifc": "skipped", "ids": "skipped", "speckle_send": "skipped", "speckle_receive": "skipped"}
print("workspace:", WORKSPACE)
print("ifcopenshell:", HAVE_IFC, " ifctester:", HAVE_IDS, " speckle tools:", "speckle_send_object" in TOOLS)

In [ ]:
# Step 1: the snapshot side. One snapshot_compute call gives counts, type names and
# the FireRating parameter per category.
snapshot = await call("snapshot_take")
snapshot_id = snapshot["snapshot_id"]
model = await call("snapshot_compute", {"snapshot_id": snapshot_id, "steps": [
    {"where": {"field": "category", "op": "in", "value": ["OST_Walls", "OST_Doors"]},
     "group_by": ["category"],
     "aggregate": {"count": "count",
                   "types": {"op": "distinct", "field": "type_name"},
                   "fire_ratings": {"op": "list", "field": "params.FireRating"}}},
]})
snapshot_rows = {row["category"]: row for row in model["rows"]}
report["snapshot_elements"] = snapshot["elements_count"]
for row in model["rows"]:
    print(row)

## Step 2: export to IFC

In Revit, `revit_export` with format `ifc` writes the file. It is a write tool, so it goes through a plan that a person approves. Mock mode cannot export, so the notebook drafts the plan, leaves it `pending`, and copies the test fixture `tiny.ifc` (3 walls, 1 door) and its IDS file into the workspace instead. The IFC tools only read files inside the workspace.

In [ ]:
export_plan = await call("plan_actions", {"actions": [{
    "tool": "revit_export",
    "arguments": {"format": "ifc", "path": str(WORKSPACE / "model.ifc")},
}]})
assert export_plan["state"] == "pending"
print("export plan:", export_plan["plan_id"], export_plan["state"])

fixtures = next(
    (base / "packages/mcp-server-revit/tests/data/ids" for base in [Path.cwd(), *Path.cwd().parents]
     if (base / "packages/mcp-server-revit/tests/data/ids/tiny.ifc").is_file()),
    None,
)
if HAVE_IFC and fixtures is not None:
    ifc_path = shutil.copy(fixtures / "tiny.ifc", WORKSPACE / "tiny.ifc")
    ids_path = shutil.copy(fixtures / "tiny.ids", WORKSPACE / "tiny.ids")
    print("IFC for the next steps:", Path(ifc_path).name)
else:
    ifc_path = None
    print("SKIPPED: IFC steps need ifcopenshell and the repository's tests/data/ids fixture.")

In [ ]:
# Step 3: read and check the IFC.
if ifc_path:
    meta = await call("ifc_get_metadata", {"ifc_path": str(ifc_path)})
    print("schema:", meta["schema"], " project:", meta["project_name"])
    schema_check = await call("ifc_validate", {"ifc_path": str(ifc_path)})
    print("schema valid:", schema_check["is_valid"], " errors:", schema_check["error_count"])
    for error in schema_check["errors"][:3]:
        print("  ", error["element_class"], error["message"].splitlines()[-1].strip())
    report["ifc"] = {"schema": meta["schema"], "schema_valid": schema_check["is_valid"]}
    if HAVE_IDS:
        ids = await call("ids_validator_validate_ifc", {"ifc_path": str(ifc_path), "ids_path": str(ids_path)})
        print("IDS:", ids["summary"])
        for spec in ids["specifications"]:
            failing = [entity["name"] for entity in spec["failed_entities"]]
            print(f"   {spec['status']:4} {spec['name']}", failing or "")
        report["ids"] = {"passed": ids["specifications_passed"], "total": ids["specifications_total"]}
    else:
        print("SKIPPED: IDS check needs ifctester (uv sync --extra ids).")
else:
    print("SKIPPED: no IFC.")

## Step 4: compare the IFC with the snapshot

Revit categories map to IFC classes (`OST_Walls` to `IfcWall`, `OST_Doors` to `IfcDoor`). The comparison counts elements per class on both sides and compares one property, `FireRating` (Revit parameter, IFC `Pset_WallCommon`). The fixture is not an export of the mock model, so the counts differ on purpose: this is what a mismatch report looks like.

In [ ]:
CLASS_FOR = {"OST_Walls": "IfcWall", "OST_Doors": "IfcDoor"}
if ifc_path:
    comparison = []
    for category, ifc_class in CLASS_FOR.items():
        found = await call("ifc_query_elements", {"ifc_path": str(ifc_path), "ifc_class": ifc_class})
        in_model = snapshot_rows.get(category, {}).get("count", 0)
        comparison.append({"category": category, "ifc_class": ifc_class, "snapshot": in_model,
                           "ifc": len(found["elements"]), "match": in_model == len(found["elements"])})
        if ifc_class == "IfcWall":
            walls = found["elements"]
    for row in comparison:
        print(f"{row['category']:10} {row['ifc_class']:8} snapshot={row['snapshot']} ifc={row['ifc']} match={row['match']}")

    ifc_ratings = []
    for wall in walls:
        props = await call("ifc_get_properties", {"ifc_path": str(ifc_path), "element_id": wall["guid"]})
        ifc_ratings.append(props["property_sets"].get("Pset_WallCommon", {}).get("FireRating"))
    model_ratings = snapshot_rows.get("OST_Walls", {}).get("fire_ratings", [])
    print("FireRating in snapshot:", model_ratings, " in IFC:", ifc_ratings)
    report["ifc"]["comparison"] = comparison
    report["ifc"]["fire_rating"] = {"snapshot": model_ratings, "ifc": ifc_ratings}

    # revit_ifc_check runs the same comparison against the live model. In mock mode it
    # reads the IFC side only and says so: verdict UNVERIFIED.
    check = await call("revit_ifc_check", {"path": str(ifc_path)})
    print("revit_ifc_check:", check["verdict"], check["ifc"]["by_class"])
else:
    print("SKIPPED: no IFC to compare.")

## Step 5: Speckle send and receive

`speckle_auth_status` is offline. `speckle_send_object` uploads data and creates a Speckle version, so it is not read-only and goes through a plan. The notebook drafts that plan and leaves it `pending`. `speckle_receive_object` needs a Speckle server and a local account; it is skipped here. `PROJECT_ID` is a placeholder.

In [ ]:
if "speckle_send_object" in TOOLS:
    auth = await call("speckle_auth_status")
    print("speckle authorized:", auth["authorized"])
    payload = {"source": "AEC Model Bridge mock snapshot", "snapshot_id": snapshot_id,
               "counts": {category: row["count"] for category, row in snapshot_rows.items()}}
    send_plan = await call("plan_actions", {"actions": [{
        "tool": "speckle_send_object",
        "arguments": {"project_id": "PROJECT_ID", "model_name": "exchange", "data": payload,
                      "message": "Mock round trip"},
    }]})
    assert send_plan["state"] == "pending"
    print("send plan:", send_plan["plan_id"], send_plan["state"])
    report["speckle_send"] = send_plan["state"]
else:
    print("SKIPPED: the Speckle provider is not loaded.")

# Requires a Speckle server: skipped in this notebook.
receive_arguments = {"project_id": "PROJECT_ID", "version_id": "VERSION_ID"}
print("SKIPPED (requires a Speckle server): speckle_receive_object", receive_arguments)
print(json.dumps(report, indent=2))

## Stop: a person approves

Both plans stay `pending` until a person approves or rejects them in the Revit panel or with the `aec-model-bridge-approve` command. AI clients cannot call the approve tool. A real export, a real IFC comparison against a live model and a real Speckle send or receive were not run here and are UNVERIFIED.